# 07 — sealed test와 RouterArena 외부 정렬평가

> **선택 실행용 실험 기반 코드입니다.** 최신 최종 실험 명세 전체를 구현한 상태는 아닙니다. 실제 본 실험 전 [`docs/ml/README.md`의 구현 차이](../../../docs/ml/README.md#implementation-gaps)를 확인하고 필요한 보완을 완료하세요. 이 노트북의 설정을 최종 연구 기준으로 사용하지 않습니다.

final validation lock을 확인한 뒤에만 test를 엽니다. test에서는 penalty를 다시 고르지 않습니다. RouterArena는 label alignment 보조 진단이며 cost-quality 우승자 선정에 사용하지 않습니다. T4 런타임을 권장합니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
archive='/content/drive/MyDrive/LLM_Router/colab-inputs/routing-objective-colab-code.tar.gz'
!rm -rf /content/LLM_Router
!tar -xzf "{archive}" -C /content
%cd /content/LLM_Router
!pip -q install -r ml/requirements-comparison.lock
!pip -q install 'transformers==4.48.3' 'safetensors>=0.4'
!pip -q install --no-deps -e ml
import json, os, pathlib, shutil, subprocess, torch
ROOT=pathlib.Path('/content/drive/MyDrive/LLM_Router/routing-objective-comparison/v1')
LOCK=ROOT/'selection/final-lock.json'
if not LOCK.is_file(): raise RuntimeError('06 notebook의 final lock을 먼저 완료하세요.')
lock=json.loads(LOCK.read_text()); assert lock['stage']=='final' and not lock['test_data_accessed']
print('locked winner:',lock['winner_method'])


In [ ]:
assert torch.cuda.is_available(), '네 모델 test 추론에는 T4 GPU 런타임을 권장합니다.'
TEST_OUT=pathlib.Path('/content/sealed-test')
if TEST_OUT.exists(): shutil.rmtree(TEST_OUT)
subprocess.run(['routing-objective-comparison','--config','ml/config/routing_objective_comparison_v1.json','evaluate-test','--validation-lock-path',str(LOCK),'--run-root',str(ROOT/'runs'),'--routerbench-test-path',str(ROOT/'dataset/test_routerbench.parquet'),'--xroutebench-test-path',str(ROOT/'dataset/test_xroutebench.parquet'),'--output-dir',str(TEST_OUT)],check=True)
test_summary=json.loads((TEST_OUT/'summary.json').read_text())
print(json.dumps(test_summary,indent=2,ensure_ascii=False)[:12000])


In [ ]:
# RouterArena를 새로 고정 revision에서 내려받아 objective train과 overlap을 제거합니다.
!routerarena-preprocessing download --data-dir ml/data/raw/routerarena --split full
!routerarena-preprocessing preprocess --data-dir ml/data/raw/routerarena --output-dir /content/routerarena-evaluation-full --split full --seed 42 --training-path {ROOT}/dataset/train.parquet
ARENA_OUT=pathlib.Path('/content/routerarena-alignment')
if ARENA_OUT.exists(): shutil.rmtree(ARENA_OUT)
subprocess.run(['routing-objective-comparison','--config','ml/config/routing_objective_comparison_v1.json','evaluate-routerarena','--validation-lock-path',str(LOCK),'--run-root',str(ROOT/'runs'),'--routerarena-path','/content/routerarena-evaluation-full/evaluation.parquet','--output-dir',str(ARENA_OUT)],check=True)
arena_summary=json.loads((ARENA_OUT/'summary.json').read_text())
print(json.dumps(arena_summary,indent=2,ensure_ascii=False))


In [ ]:
for source,name in [(TEST_OUT,'sealed-test'),(ARENA_OUT,'routerarena-alignment')]:
    final=ROOT/'evaluation'/name; staging=pathlib.Path(str(final)+'.staging')
    if staging.exists(): shutil.rmtree(staging)
    staging.parent.mkdir(parents=True,exist_ok=True); shutil.copytree(source,staging)
    if final.exists(): raise RuntimeError(f'이미 봉인된 평가를 덮어쓸 수 없습니다: {final}')
    os.replace(staging,final)
print('최종 평가 게시 완료:',ROOT/'evaluation')
